<a href="https://colab.research.google.com/github/Rohil121/bharat-portfolio-lab/blob/v0.4-forecasting-risk-models/notebooks/04_forecasting_risk_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bharat Portfolio Lab v0.4 — Forecasting and Risk Models

## Objective

This notebook develops forecasting and volatility-risk models for Indian
listed equities.

The v0.4 framework will include:

- ARIMA return forecasting
- naive benchmark forecasts
- forecast-confidence intervals
- stationarity and residual diagnostics
- ARCH/GARCH volatility forecasting
- forecast evaluation using a chronological test period
- dynamic volatility forecasts for portfolio allocation

## Research principle

Forecasting models will be evaluated against simple benchmarks rather than
assumed to be useful.

The objective is to quantify uncertainty and forecast risk, not to claim
that tomorrow's stock price can be predicted accurately.

## Dynamic-portfolio requirement

All forecasting functions must accept arbitrary NSE/BSE ticker inputs so
that they can later support user-selected portfolios in the final
application.

In [1]:
%pip install -q yfinance statsmodels arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 13.0 MB/s eta 0:00:00


In [2]:
# ---------------------------------------------------------
# v0.4 environment setup
# ---------------------------------------------------------

from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

from arch import arch_model
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
)

from statsmodels.graphics.tsaplots import (
    plot_acf,
    plot_pacf,
)

from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    het_arch,
)

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

TRADING_DAYS = 252
RISK_FREE_RATE = 0.065

DEFAULT_BENCHMARK_TICKER = "^NSEI"
DEFAULT_BENCHMARK_NAME = "Nifty 50"

DEFAULT_FORECAST_TICKERS = [
    "HDFCBANK.NS",
    "TCS.NS",
    "HINDUNILVR.NS",
    "SUNPHARMA.NS",
    "POWERGRID.NS",
    "BHARTIARTL.NS",
    "LT.NS",
    "M&M.NS",
    "BEL.NS",
    "TRENT.NS",
]

FORECAST_DATA_START = "2016-01-01"
FORECAST_TEST_START = pd.Timestamp("2024-01-01")

print("v0.4 forecasting environment prepared.")
print("Default equities:", len(DEFAULT_FORECAST_TICKERS))
print("Forecast test period begins:", FORECAST_TEST_START.date())

v0.4 forecasting environment prepared.
Default equities: 10
Forecast test period begins: 2024-01-01


## 1. Market Data and Chronological Split

Daily adjusted closing prices are downloaded for the selected Indian
equities and benchmark.

Forecasting models will be trained using observations before
1 January 2024 and evaluated only on observations from 1 January 2024
onward.

Daily log returns are used for return forecasting because stock-price
levels are generally non-stationary.

The data function accepts arbitrary ticker lists so it can later support
user-selected portfolios in the Streamlit application.

In [3]:
# ---------------------------------------------------------
# Reusable Indian market-data loader
# ---------------------------------------------------------

def download_adjusted_close_prices(
    tickers,
    benchmark_ticker="^NSEI",
    start_date="2016-01-01",
    end_date=None,
):
    """
    Download adjusted closing prices for an arbitrary list of
    NSE/BSE securities and an Indian market benchmark.
    """

    if not tickers:
        raise ValueError(
            "At least one equity ticker must be supplied."
        )

    cleaned_tickers = [
        str(ticker).strip().upper()
        for ticker in tickers
        if str(ticker).strip()
    ]

    all_tickers = list(
        dict.fromkeys(
            cleaned_tickers
            + [benchmark_ticker]
        )
    )

    if end_date is None:
        end_date = (
            pd.Timestamp.today().normalize()
            + pd.Timedelta(days=1)
        )

    raw_data = yf.download(
        tickers=all_tickers,
        start=str(pd.Timestamp(start_date).date()),
        end=str(pd.Timestamp(end_date).date()),
        auto_adjust=True,
        progress=False,
        threads=True,
    )

    if raw_data.empty:
        raise ValueError(
            "No market data was returned."
        )

    # Handle Yahoo Finance's possible column structures
    if isinstance(raw_data.columns, pd.MultiIndex):

        if "Close" in raw_data.columns.get_level_values(0):

            close_prices = (
                raw_data["Close"]
                .copy()
            )

        elif "Close" in raw_data.columns.get_level_values(1):

            close_prices = (
                raw_data
                .xs(
                    "Close",
                    axis=1,
                    level=1,
                )
                .copy()
            )

        else:
            raise ValueError(
                "Closing-price data was not returned."
            )

    else:

        if "Close" not in raw_data.columns:
            raise ValueError(
                "Closing-price data was not returned."
            )

        close_prices = raw_data[["Close"]].copy()

        if len(all_tickers) == 1:
            close_prices.columns = all_tickers

    close_prices = (
        close_prices
        .reindex(columns=all_tickers)
        .sort_index()
        .dropna(how="all")
    )

    missing_tickers = [
        ticker
        for ticker in all_tickers
        if (
            ticker not in close_prices.columns
            or close_prices[ticker].dropna().empty
        )
    ]

    if missing_tickers:
        raise ValueError(
            "No usable price history was found for: "
            + ", ".join(missing_tickers)
        )

    return close_prices

In [4]:
# ---------------------------------------------------------
# Download forecasting dataset
# ---------------------------------------------------------

forecast_price_data = download_adjusted_close_prices(
    tickers=DEFAULT_FORECAST_TICKERS,
    benchmark_ticker=DEFAULT_BENCHMARK_TICKER,
    start_date=FORECAST_DATA_START,
)

forecast_stock_prices = (
    forecast_price_data[
        DEFAULT_FORECAST_TICKERS
    ]
    .copy()
)

forecast_benchmark_prices = (
    forecast_price_data[
        DEFAULT_BENCHMARK_TICKER
    ]
    .rename(DEFAULT_BENCHMARK_NAME)
)

print("Forecasting market data downloaded.")
print("-" * 60)
print(
    "Available period:",
    forecast_price_data.index.min().date(),
    "to",
    forecast_price_data.index.max().date(),
)
print(
    "Securities:",
    len(DEFAULT_FORECAST_TICKERS),
)
print(
    "Benchmark:",
    DEFAULT_BENCHMARK_NAME,
)

Forecasting market data downloaded.
------------------------------------------------------------
Available period: 2016-01-01 to 2026-07-30
Securities: 10
Benchmark: Nifty 50


In [5]:
# ---------------------------------------------------------
# Calculate log returns and chronological split
# ---------------------------------------------------------

forecast_log_returns = (
    np.log(
        forecast_price_data
        / forecast_price_data.shift(1)
    )
)

development_log_returns = (
    forecast_log_returns.loc[
        forecast_log_returns.index
        < FORECAST_TEST_START
    ]
)

test_log_returns = (
    forecast_log_returns.loc[
        forecast_log_returns.index
        >= FORECAST_TEST_START
    ]
)

assert not development_log_returns.empty
assert not test_log_returns.empty
assert (
    development_log_returns.index.max()
    < FORECAST_TEST_START
)
assert (
    test_log_returns.index.min()
    >= FORECAST_TEST_START
)

print("Chronological forecasting split completed.")
print("-" * 60)
print(
    "Development period:",
    development_log_returns.index.min().date(),
    "to",
    development_log_returns.index.max().date(),
)
print(
    "Test period:",
    test_log_returns.index.min().date(),
    "to",
    test_log_returns.index.max().date(),
)

Chronological forecasting split completed.
------------------------------------------------------------
Development period: 2016-01-01 to 2023-12-29
Test period: 2024-01-01 to 2026-07-30


In [6]:
# ---------------------------------------------------------
# Data-quality summary by security
# ---------------------------------------------------------

data_quality_records = []

for ticker in forecast_price_data.columns:

    ticker_prices = (
        forecast_price_data[ticker]
        .dropna()
    )

    ticker_development_returns = (
        development_log_returns[ticker]
        .dropna()
    )

    ticker_test_returns = (
        test_log_returns[ticker]
        .dropna()
    )

    data_quality_records.append(
        {
            "Ticker": ticker,
            "First Price Date":
                ticker_prices.index.min().date(),

            "Latest Price Date":
                ticker_prices.index.max().date(),

            "Price Observations":
                len(ticker_prices),

            "Development Returns":
                len(ticker_development_returns),

            "Test Returns":
                len(ticker_test_returns),

            "Missing Price Values":
                forecast_price_data[ticker]
                .isna()
                .sum(),
        }
    )


forecast_data_quality = (
    pd.DataFrame(data_quality_records)
    .set_index("Ticker")
)

display(forecast_data_quality)

assert (
    forecast_data_quality[
        "Development Returns"
    ] >= 500
).all()

assert (
    forecast_data_quality[
        "Test Returns"
    ] >= 100
).all()

,First Price Date,Latest Price Date,Price Observations,Development Returns,Test Returns,Missing Price Values
Ticker,,,,,,
HDFCBANK.NS,2016-01-01,2026-07-30,2615,1974,640,0
TCS.NS,2016-01-01,2026-07-30,2615,1974,640,0
HINDUNILVR.NS,2016-01-01,2026-07-30,2615,1974,640,0
SUNPHARMA.NS,2016-01-01,2026-07-30,2615,1974,640,0
POWERGRID.NS,2016-01-01,2026-07-30,2615,1974,640,0
BHARTIARTL.NS,2016-01-01,2026-07-30,2615,1974,640,0
LT.NS,2016-01-01,2026-07-30,2615,1974,640,0
M&M.NS,2016-01-01,2026-07-30,2615,1974,640,0
BEL.NS,2016-01-01,2026-07-30,2615,1974,640,0
